# 05 · Moseley's law: K edges and the Rydberg constant

EPN Práctica 5 (and LD's "Moseley's law" experiment) on the 554 800: scan the NaCl crystal in COUPLED mode once
without an absorber and once behind each foil (Zr, Mo, Ag, In). The transmission T = R(foil)/R(open) rises at the
foil's K edge. Bragg's law turns each edge angle into a wavelength λ_K, and Moseley's law
1/λ_K = R (Z − σ_K)² gives the Rydberg constant R and the K screening constant σ_K. The analysis lives in
`braggsim.analysis` (`k_edge_deg`, `edge_wavelength_table`, `moseley_fit`); this notebook runs it and shows the result.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xraylib

from braggsim.analysis import edge_wavelength_table, moseley_fit
from braggsim.constants import HC_KEV_PM, R_INF_PER_M
from braggsim.crystal import theta_from_lambda
from braggsim.detector import sample_counts
from braggsim.filters import FOILS_MM, areal_density_mg_cm2
from braggsim.scan import coupled_betas, expected_rate

plt.rcParams.update(
    {
        "axes.grid": True,
        "grid.color": "#e5e5e5",
        "lines.linewidth": 1.2,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
DATA, MODEL, ALT = "#1a1a19", "#2a78d6", "#eb6834"

## Five simulated scans

Práctica 5 settings: U = 35 kV, I = 1 mA, COUPLED 3.7° → 7.5°, Δβ = 0.1°, Δt = 5 s. Poisson noise is the only random
step (fixed seed).

In [ ]:
DT_S = 5.0
Z = {"Zr": 40, "Mo": 42, "Ag": 47, "In": 49}
beta = coupled_betas(3.7, 7.5, 0.1)
rng = np.random.default_rng(2026)


def measure(filters=()):
    return sample_counts(expected_rate(beta, 35.0, 1.0, filters=filters), DT_S, rng) / DT_S


r_open = measure()
r_foil = {el: measure(((el, areal_density_mg_cm2(el, FOILS_MM[el])),)) for el in Z}
rows = edge_wavelength_table(beta, r_open, r_foil, DT_S)

## Transmission and the edge

Dashed orange: the angle found from the data. Grey: θ(λ_K) from the tabulated xraylib edge. The found edge sits
0.006–0.045° below it. The counter sees both spectra blurred by the instrument width (σ = 0.12°), and a ratio of two
blurred spectra is not a blurred ratio: the steep open spectrum next to the Mo lines (Zr beside Kα, Mo beside Kβ)
pulls the edge the most. Two smaller contributions: the interpolation of μ/ρ on its wavelength grid (a ramp of
about ±0.009° around the edge) and the dead time of the observed rates (≤ 0.006°). Leak, scatter and background are
negligible (< 0.0003°). This is real instrument physics and is not corrected. The four foil scans share one open
scan, so their edges are slightly correlated; `moseley_fit` treats them as independent.

In [ ]:
lam_tab = {el: HC_KEV_PM / xraylib.EdgeEnergy(z, xraylib.K_SHELL) for el, z in Z.items()}
fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
for ax, el, row in zip(axes.flat, Z, rows, strict=True):
    ax.plot(beta, r_foil[el] / r_open, ".", color=DATA, ms=4)
    ax.axvline(row["theta_deg"], color=ALT, ls="--", lw=1)
    ax.axvline(float(theta_from_lambda(lam_tab[el])), color="#999", lw=1)
    ax.set_title(f"{el} (Z = {Z[el]})", fontsize=10)
for ax in axes[1]:
    ax.set_xlabel("β (°)")
for ax in axes[:, 0]:
    ax.set_ylabel("T = R_foil / R_open")
fig.tight_layout()

## The edges

In [ ]:
table = pd.DataFrame(rows).set_index("element")
table["lambda_tab_pm"] = pd.Series(lam_tab)
table["theta_tab_deg"] = [float(theta_from_lambda(lam_tab[el])) for el in table.index]
table.round(3)

## Moseley's law

√(1/λ_K) is a straight line in Z: slope √R, intercept −√R σ_K.

In [ ]:
z = np.array(list(Z.values()))
lam = np.array([r["lambda_pm"] for r in rows])
lam_err = np.array([r["lambda_err_pm"] for r in rows])
R, R_err, sigma, sigma_err, chi2 = moseley_fit(z, lam, lam_err)
R_tab, R_tab_err, sigma_tab, sigma_tab_err, _ = moseley_fit(z, list(lam_tab.values()))

zz = np.linspace(38, 51, 50)
fig, ax = plt.subplots(figsize=(6, 4))
y = np.sqrt(1e12 / lam) * 1e-5  # 1e5 m^-1/2
ax.errorbar(z, y, yerr=0.5 * y * lam_err / lam,
            fmt="o", color=DATA, label="measured edges")  # fmt: skip
ax.plot(zz, np.sqrt(R) * (zz - sigma) * 1e-5, color=MODEL, label="fit")
ax.set_xlabel("Z")
ax.set_ylabel("√(1/λ_K) (10⁵ m⁻¹ᐟ²)")
ax.legend()
fig.tight_layout()

## Comparison

Rows: this measurement, the fit of the tabulated xraylib edges, LD's published example and the Rydberg constant
R∞ (CODATA 2022). χ²/dof is the weighted misfit of the line to the four edges, with the counting errors only.

In [ ]:
pd.DataFrame(
    {
        "R (1e7 1/m)": [
            f"{R / 1e7:.3f} ± {R_err / 1e7:.3f}",
            f"{R_tab / 1e7:.3f}",
            "1.056",
            f"{R_INF_PER_M / 1e7:.3f}",
        ],
        "sigma_K": [f"{sigma:.2f} ± {sigma_err:.2f}", f"{sigma_tab:.2f}", "2.7", ""],
        "chi2/dof": [f"{chi2:.1f}", "", "", ""],
    },
    index=["this measurement", "tabulated edges", "LD example", "R_inf"],
)  # fmt: skip

## Things to read correctly

- **R is below R∞ (physics).** The tabulated edges give R = 0.987 R∞, 1.3 % low, and σ_K ≈ 3.4. Moseley's formula is
  the hydrogen-like limit and the straight line absorbs two Z-dependent effects of opposite sign: relativistic binding
  alone would raise R (a Dirac 1s fit with constant σ gives ≈ 1.07 R∞), and outer-electron screening that grows with Z
  outweighs it. A student who finds R a few % low has measured correctly. (LD's "lit. 3.6" for σ_K is the kind of value
  one gets with R fixed at R∞; in a free fit R and σ_K are strongly correlated, so compare 3.4 with 3.6 with care.)
- **The edges are biased low by 0.006–0.045° (instrument).** The shift differs between foils (Zr and Mo move most), so
  the counting-weighted fit lands ≈ 2.3 % low in R and ≈ 0.6 low in σ_K against the tabulated edges. The quoted errors
  are counting statistics only and do not include it: a measurement scatters by them around the noise-free fit, not
  around the tabulated one. Noise-free χ²/dof ≈ 0.2, so at P5 counting errors the four edges are consistent with a line.
  Both are documented in `model/PARAMETERS.md`, "Moseley analysis".